# Introduction

## Limitation of our previous RAG

- No permanant storage of vector database
    - This will lead to suboptimal use of RAM as all the embeddings get stored in the RAM only
- Uses `Symentic Search` (Vector Similarity Search) only
    - Symentic search means just finding nearest neighbors 
    - The accuracy of the answer (result) is limited due to this reason

## Improvements in this RAG

- Permanant storage of chromadb collection
- Introduction of `Keyword Search` along with `Symentic Search`
- We will understand the use of `metadata`

### Re-ranking the context
- Re-raking models are used to rank the chunks based on their relevancy before feeding them to the LLM

In [1]:
from openai import OpenAI
from dotenv import load_dotenv
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
import chromadb
import os
from rank_bm25 import BM25Okapi
import numpy as np

C:\Users\alakh\AppData\Roaming\Python\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv()

client = OpenAI(

    base_url="https://openrouter.ai/api/v1",

    api_key=os.getenv("OPENROUTER_API_KEY")
)

# ---------------- LOAD PDF ----------------
pdf_path = "gk-book.pdf"

reader = PdfReader(pdf_path)

document_text = ""

for page in reader.pages:
    document_text += page.extract_text()

print("\nPdf loaded successfully...\n")

# ----------------------- Chunking -----------------------

chunk_size = 1000

chunks = []

for i in range(0, len(document_text), chunk_size):      # i = 0, 1000, 2000, 3000....
    chunks.append(document_text[i : i+chunk_size])

print(f"\nTotal chunks created = {len(chunks)}\n")


# ----------------------- Vector Embedding Model -----------------------

embedding_model = SentenceTransformer(
    model_name_or_path= "all-MiniLM-L6-v2"
    # model_name_or_path= "multi-qa-mpnet-base-cos-v1"
)

print("\nEmbedding Model Instance created\n")


Pdf loaded successfully...


Total chunks created = 469



Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2243.72it/s]



Embedding Model Instance created



In [6]:
# ----------------- Creating Vector Database with Permanant Storage -----------------

chroma_client = chromadb.PersistentClient()

collection = chroma_client.get_or_create_collection(
    
    name= "enterprise_rag_database"

)
print("Vector database created/retrieved successfully...")

Vector database created/retrieved successfully...


In [7]:
# -------------- Storing Vector Embeddings In Database Along With Metadatas --------------
for index, chunk in enumerate(chunks):

    embedding = embedding_model.encode(chunk)

    collection.upsert(
        ids= [str(index)],

        embeddings= [embedding],

        documents= [chunk],

        metadatas= [
            {
                "chunk_id" : index
            }
        ]
    )

print("\nEmbeddings are stored in the vector database successfully...\n")



Embeddings are stored in the vector database successfully...



In [5]:
# ----------------------- BM25 Setup -----------------------

# To perform keyword search, we will first need to identify all the words of each chunk
tokenized_chunks = [chunk.split() for chunk in chunks]

# print("Tokenized Chunks:\n", np.array(tokenized_chunks[0]))

bm25 = BM25Okapi(corpus=tokenized_chunks)

In [ ]:
# ----------------------- Conversation Memory -----------------------
messages = [
    {
        "role" : "system",
        "content" : """
        You are a helpful AI document assitant. 

        Answer the user's question ONLY from the provided context. If answer is not available in the context, simply say: "I could not find the answer in the document(s)."
        """
    }
]


# ------------- OpenAI Agent -------------
load_dotenv()

client = OpenAI(

    base_url= "https://openrouter.ai/api/v1",

    api_key= os.getenv("OPENROUTER_API_KEY")

)


In [ ]:
# ------------- Chat Loop -------------

print("Enterprise RAG Started!!!")
print("Type 'exit' whenever you want to quit.")

while True:
    # ------------- User's Question -------------
    question = input('\n[You] : ')

    if question.lower() == "exit":
        break

    # ------------- Vector Embeddings Of question -------------
    question_embedding = embedding_model.encode(question)

    # ------------- Vector Similarity Search -------------
    similarity_result = collection.query(
        query_embeddings= [question_embedding],

        n_results= 5
    )

    # ------------- Keyword Searching -------------
    tokenized_query = question.split()

    bm25_result = bm25.get_top_n(
        query= tokenized_query,

        documents= chunks,

        n= 5
    )
    
    
    # ------------- Generating Context/Hybrid Search -------------
    final_chunks = list(
        set(similarity_result + bm25_result)
    )

    # ------------- Re-ranking the chunks in the context -------------



    